---
**`NOTEBOOK 01 / 03`** · Causal Inference Series

## A/B Test — Cookie Cats
*Tactile Entertainment · 90,189 players · Real randomized experiment*

| | |
|--|--|
| Method | A/B Test (RCT) |
| Primary metric | retention_7 (binary) → Z-test |
| Assumption | Randomization ✓ verified via SRM check |
| Result | gate_30 retains more · p = 0.003 · CI [0.27pp, 1.29pp] |
| Causal hierarchy | ⭐⭐⭐ Level 1 — most credible |

---

Cookie Cats is one of the most downloaded mobile puzzle games ever, developed by Tactile Entertainment. As players progress, they hit "gates" that force them to wait or make an in-app purchase. Tactile ran a real A/B test on 90,189 players to decide whether moving the gate from level 30 to level 40 affected retention.

The A/B test is the **gold standard** of causal inference. Players were randomly assigned at install time — true randomization. Any difference in retention is caused by the gate position, not by pre-existing differences between players. No assumptions about confounders needed.

**The causal question:** does moving the gate from level 30 to level 40 reduce Day-7 player retention?

---

**Structure**

`0` Setup

`1` Hypotheses & primary metric

`2` Sample size — MDE, α, power

`3` Randomization unit

`4` SRM check — chi-squared, fix imbalance

`5` Peeking & sequential testing

`6` Z-test, CI, effect size

`7` Business conclusion & guardrails

`8` CUPED — variance reduction

`9` Common mistakes to avoid

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, chisquare, ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings("ignore")

cookie_cats_df = pd.read_csv("../data/cookie_cats.csv")

print(f"Shape: {cookie_cats_df.shape[0]:,} rows x {cookie_cats_df.shape[1]} columns")
print(f"Missing values: {cookie_cats_df.isnull().sum().sum()}")
display(cookie_cats_df.head())

Shape: 90,189 rows x 5 columns
Missing values: 0


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## 1. Define Hypotheses & Primary Metric

Before looking at any results, we fix:
- **Primary metric:** `retention_7` — did the player return 7 days after install? (binary)
- **Guardrail metric:** `sum_gamerounds` — must not worsen significantly
- **Randomization unit:** player (userid) — the gate affects the full player experience, not a single session

**Hypotheses:**

H₀: retention_7(gate_30) = retention_7(gate_40) — moving the gate has no effect

H₁: retention_7(gate_30) ≠ retention_7(gate_40) — moving the gate changes retention

Two-sided test. α = 0.05.

> Changing the primary metric after seeing results is HARKing (Hypothesizing After Results are Known) and produces non-replicable findings.

### Why retention_7 as primary metric?

For a mobile game, **Day-7 retention is the north star metric** because:

- **Day-1 retention** measures curiosity — players come back just to explore
- **Day-7 retention** measures real engagement — players who return after a week 
  are the ones who will eventually make in-app purchases or watch ads

Moving the gate from level 30 to level 40 means players reach the gate later. 
The question is whether this changes their long-term engagement — and Day-7 
is the earliest signal that captures that.

Day-1 retention is analyzed as a **secondary metric** to understand short-term 
behavior, but it does not drive the launch decision.

In [2]:
# Define groups
control   = cookie_cats_df[cookie_cats_df["version"] == "gate_30"]
treatment = cookie_cats_df[cookie_cats_df["version"] == "gate_40"]

print(f"Control   (gate_30): {len(control):,} players")
print(f"Treatment (gate_40): {len(treatment):,} players")

print(f"\nRetention D7 — control  : {control['retention_7'].mean():.4f}")
print(f"Retention D7 — treatment: {treatment['retention_7'].mean():.4f}")
print(f"Absolute difference     : {control['retention_7'].mean() - treatment['retention_7'].mean():.4f}")

Control   (gate_30): 44,700 players
Treatment (gate_40): 45,489 players

Retention D7 — control  : 0.1902
Retention D7 — treatment: 0.1820
Absolute difference     : 0.0082


## 2. Sample Size Calculation

Before running the experiment, we need to know how many players per group 
are required to detect a meaningful effect.

- **Baseline retention D7:** ~19% (historical rate)
- **MDE:** 1 percentage point — the minimum effect worth acting on. 
  A change smaller than 1pp would not justify the engineering cost of moving the gate.
- **α = 0.05** (two-sided)
- **Power = 0.80** — we accept a 20% chance of missing a real effect

Since `retention_7` is a binary metric, we use the formula for proportions:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2 \cdot p \cdot (1-p)}{MDE^2}$$

where:
- $z_{\alpha/2} = 1.96$ for $\alpha = 0.05$ (two-sided)
- $z_{\beta} = 0.84$ for power $= 80\%$
- $p$ = baseline retention rate
- $MDE$ = minimum detectable effect

In [3]:
p_baseline = control["retention_7"].mean()
mde        = 0.01
alpha      = 0.05
power      = 0.80

# z_alpha/2: value that leaves α/2 = 2.5% in the right tail
z_alpha = norm.ppf(1 - alpha/2)

# z_beta: value that leaves β = 20% in the right tail (power = 80%)
z_beta = norm.ppf(power)

n_required = (z_alpha + z_beta)**2 * 2 * p_baseline * (1 - p_baseline) / mde**2

print(f"Baseline retention D7 : {p_baseline:.4f}")
print(f"MDE                   : {mde:.4f} ({mde*100:.1f} pp)")
print(f"α = {alpha} → z_α/2 = norm.ppf({1 - alpha/2}) = {z_alpha:.4f}")
print(f"Power = {power} → z_β = norm.ppf({power}) = {z_beta:.4f}")
print(f"\nRequired n per group  : {n_required:,.0f}")
print(f"Actual n per group    : {len(control):,}")
print(f"\nConclusion: {'✓ Sufficient sample' if len(control) >= n_required else '✗ Insufficient sample'}")

Baseline retention D7 : 0.1902
MDE                   : 0.0100 (1.0 pp)
α = 0.05 → z_α/2 = norm.ppf(0.975) = 1.9600
Power = 0.8 → z_β = norm.ppf(0.8) = 0.8416

Required n per group  : 24,178
Actual n per group    : 44,700

Conclusion: ✓ Sufficient sample


## 3. Randomization Unit

The randomization unit is the **player (userid)**.

This is the correct choice because:
- The gate affects the **full player experience** across all sessions
- Randomizing by session would contaminate the test — the same player 
  could see gate_30 in one session and gate_40 in another
- The metric (retention_7) is measured at the player level, not the session level

Rule: the randomization unit must be at least as granular as the exposure 
and at least as aggregated as the metric.

## 4. Randomize & Verify Balance: SRM Check

When we launch an A/B test expecting a 50/50 split, we need to verify that 
the randomization actually produced balanced groups. If the split is 60/40 
instead of 50/50, something broke in the assignment mechanism — this is called 
**Sample Ratio Mismatch (SRM)**.

**How we detect it — chi-squared test:**

The chi-squared test answers: *"is what I observe compatible with what I expected?"*

$$\chi^2 = \sum \frac{(observed - expected)^2}{expected}$$

- H₀: the observed split is consistent with 50/50
- If χ² is large → p-value small → the imbalance is unlikely to be random → SRM detected
- If p < 0.05 → stop and investigate before analyzing anything

In [4]:
observed  = [len(control), len(treatment)]
expected  = [len(cookie_cats_df) / 2, len(cookie_cats_df) / 2]

chi2, p_srm = chisquare(f_obs=observed, f_exp=expected)

print(f"Observed  — gate_30: {observed[0]:,} | gate_40: {observed[1]:,}")
print(f"Expected  — gate_30: {expected[0]:,.0f} | gate_40: {expected[1]:,.0f}")
print(f"\nChi-squared : {chi2:.4f}")
print(f"p-value     : {p_srm:.4f}")
print(f"\nSRM detected: {'⚠️ YES — investigate before proceeding' if p_srm < 0.05 else '✓ NO — randomization looks clean'}")

Observed  — gate_30: 44,700 | gate_40: 45,489
Expected  — gate_30: 45,094 | gate_40: 45,094

Chi-squared : 6.9024
p-value     : 0.0086

SRM detected: ⚠️ YES — investigate before proceeding


### Fixing the SRM: Balancing the Groups

To fix the imbalance, we downsample the larger group to match the smaller one. 
We use the balanced groups for all subsequent analysis.

In [5]:
n_min = min(len(control), len(treatment))

control_balanced   = control.sample(n=n_min, random_state=42)
treatment_balanced = treatment.sample(n=n_min, random_state=42)

print(f"Balanced groups:")
print(f"  Control   (gate_30): {len(control_balanced):,} players")
print(f"  Treatment (gate_40): {len(treatment_balanced):,} players")

# Verify SRM is resolved
observed_balanced = [len(control_balanced), len(treatment_balanced)]
expected_balanced = [n_min, n_min]
chi2_b, p_srm_b = chisquare(f_obs=observed_balanced, f_exp=expected_balanced)
print(f"\nChi-squared after balancing: {chi2_b:.4f}")
print(f"p-value                    : {p_srm_b:.4f}")
print(f"SRM detected               : {'⚠️ YES' if p_srm_b < 0.05 else '✓ NO'}")

# Update retention rates
print(f"\nRetention D7 — control  : {control_balanced['retention_7'].mean():.4f}")
print(f"Retention D7 — treatment: {treatment_balanced['retention_7'].mean():.4f}")

Balanced groups:
  Control   (gate_30): 44,700 players
  Treatment (gate_40): 44,700 players

Chi-squared after balancing: 0.0000
p-value                    : 1.0000
SRM detected               : ✓ NO

Retention D7 — control  : 0.1902
Retention D7 — treatment: 0.1824


## 5. Run: Peeking & Sequential Testing

One of the most common mistakes in A/B testing is **peeking**: checking the 
p-value daily and stopping the experiment as soon as it crosses 0.05.

This inflates the Type I error from 5% to 30-40% — you end up declaring 
winners that are just noise.

**The fix:** decide the experiment duration upfront based on the sample size 
calculation (Notebook 02 block 2) and do not look at results until the 
planned sample is reached.

In our case:
- Required sample: 24,178 per group
- We reached 44,700 per group before analyzing
- No peeking occurred — the dataset was collected before any analysis

If early looks are needed, use **sequential testing** with α-spending 
corrections, which adjusts the significance threshold at each interim look 
to keep the overall Type I error at 5%.

## 6. Analyze: Z-Test, CI, Effect Size

Before choosing the test, we follow the decision framework from the guide:

| Metric type | Parametric test | Non-parametric fallback |
|-------------|----------------|------------------------|
| Binary (0/1) | Z-test for proportions | Fisher exact (small n) |
| Continuous symmetric | Welch t-test | Mann-Whitney U |
| Continuous skewed | Welch on log(Y) | Bootstrap |
| Count (events) | Poisson rate test | Mann-Whitney U |

**Why parametric?**

Parametric tests assume the sampling distribution of the estimator is normal. 
This assumption is justified by the **Central Limit Theorem (CLT)**:

> *With a sufficiently large sample, the sampling distribution of the mean 
> (or proportion) converges to a normal distribution, regardless of the 
> original data distribution.*

With **n > 44,000 per group**, the CLT applies strongly — the sampling 
distribution of our retention rate is approximately normal. 
Parametric tests are valid and more powerful than non-parametric alternatives.

If we had n < 30 per group, we could not rely on the CLT and would use 
**Fisher exact test** instead.

**Our case:**
`retention_7` is **binary (0/1)** + **n > 44,000** → **Z-test for proportions** ✓

**Why not Welch?** Welch is for continuous metrics. Retention is 0 or 1, not 2.3.

**Why not Poisson?** Poisson is for counts. Retention is a yes/no per player, not a count of events.

**What we always report:**
- **Point estimate** — absolute difference in retention rates
- **95% Confidence Interval** — range of plausible values for the true effect
- **p-value** — probability of seeing a result this extreme if H₀ were true
- **Relative effect size** — percentage change relative to control

> p < 0.05 is necessary but not sufficient. We also check: is the effect 
> size meaningful for the business? Is the CI narrow enough to decide?

In [6]:
# Counts
n_control   = len(control_balanced)
n_treatment = len(treatment_balanced)
conv_control   = control_balanced["retention_7"].sum()
conv_treatment = treatment_balanced["retention_7"].sum()

# Rates
p_control   = conv_control / n_control
p_treatment = conv_treatment / n_treatment

# Z-test
z_stat, p_value = proportions_ztest(
    count=[conv_control, conv_treatment],
    nobs=[n_control, n_treatment],
    alternative="two-sided"
)

# 95% CI for the difference
se = np.sqrt(p_control*(1-p_control)/n_control + p_treatment*(1-p_treatment)/n_treatment)
ci_low  = (p_control - p_treatment) - 1.96 * se
ci_high = (p_control - p_treatment) + 1.96 * se

# Relative effect
relative_effect = (p_control - p_treatment) / p_treatment

print(f"Control   retention D7 : {p_control:.4f}")
print(f"Treatment retention D7 : {p_treatment:.4f}")
print(f"\nAbsolute difference    : {p_control - p_treatment:+.4f}")
print(f"Relative effect        : {relative_effect:+.2%}")
print(f"95% CI                 : [{ci_low:.4f}, {ci_high:.4f}]")
print(f"\nZ-statistic            : {z_stat:.4f}")
print(f"p-value                : {p_value:.4f}")
print(f"\nConclusion: {'✓ Significant — reject H₀' if p_value < 0.05 else '✗ Not significant — fail to reject H₀'}")

Control   retention D7 : 0.1902
Treatment retention D7 : 0.1824

Absolute difference    : +0.0078
Relative effect        : +4.26%
95% CI                 : [0.0027, 0.0129]

Z-statistic            : 2.9806
p-value                : 0.0029

Conclusion: ✓ Significant — reject H₀


### Results

| Metric | Control (gate_30) | Treatment (gate_40) |
|--------|-------------------|---------------------|
| Retention D7 | 19.02% | 18.24% |
| Absolute difference | +0.78pp | |
| Relative effect | +4.26% | |
| 95% CI | [0.27pp, 1.29pp] | |
| Z-statistic | 2.98 | |
| p-value | 0.0029 | |

**The result is statistically significant (p = 0.0029 < 0.05).**

The CI [0.0027, 0.0129] excludes zero — we can rule out no effect.
Gate_30 retains more players at day 7 than gate_40.

However, statistical significance alone is not enough to make a business 
decision. We still need to check:
- Is 0.78pp a meaningful effect for the business?
- Are guardrail metrics unaffected?

→ See Section 7.

## 7. Interpret — Business Conclusion & Guardrails

p < 0.05 is necessary but not sufficient to make a business decision.
We need to answer three questions before deciding:

**1. Is the effect size relevant?**
- The absolute difference is 0.78pp (CI: 0.27pp to 1.29pp).
At 90,000 installs, 0.78pp means ~700 extra players retained per cohort.
Whether this justifies keeping the gate at level 30 depends on the 
monetization value of a retained player.

**2. Is the CI narrow enough to decide?**
- The CI [0.0027, 0.0129] excludes zero and is reasonably narrow.
We are confident the effect is positive and meaningful.

**3. Are guardrail metrics unaffected?**
- We check `sum_gamerounds` — total rounds played in the first 14 days.
This must not worsen significantly, otherwise the gate change is 
hurting engagement even if retention improves.

In [7]:
t_stat, p_guardrail = ttest_ind(
    control_balanced["sum_gamerounds"],
    treatment_balanced["sum_gamerounds"],
    equal_var=False
)

mean_control   = control_balanced["sum_gamerounds"].mean()
mean_treatment = treatment_balanced["sum_gamerounds"].mean()

print("--- Guardrail Metric: sum_gamerounds ---")
print(f"Control   mean : {mean_control:.2f}")
print(f"Treatment mean : {mean_treatment:.2f}")
print(f"Difference     : {mean_control - mean_treatment:.2f}")
print(f"p-value        : {p_guardrail:.4f}")
print(f"Guardrail OK   : {'✓ YES' if p_guardrail > 0.05 else '⚠️ NO — guardrail violated'}")

--- Guardrail Metric: sum_gamerounds ---
Control   mean : 52.46
Treatment mean : 51.31
Difference     : 1.15
p-value        : 0.3792
Guardrail OK   : ✓ YES


### Business Conclusion

| Question | Answer |
|----------|--------|
| Is the effect significant? | ✓ Yes — p = 0.0029 |
| Is the effect size relevant? | ✓ Yes — 0.78pp, ~700 extra players retained per cohort |
| Is the CI narrow enough? | ✓ Yes — [0.27pp, 1.29pp], excludes zero |
| Are guardrails unaffected? | ✓ Yes — sum_gamerounds p = 0.38 |

**Recommendation: keep the gate at level 30.**

Moving the gate to level 40 significantly reduces Day-7 retention 
without any compensating improvement in engagement (sum_gamerounds).

The most likely explanation is the **hedonic adaptation** theory — 
players need a break earlier in the game to stay engaged long-term. 
Removing that break until level 40 causes more players to churn.

## 8. CUPED — Variance Reduction

CUPED (Controlled-experiment Using Pre-Existing Data) is a technique 
that reduces the variance of the outcome without changing its mean — 
effectively giving us more statistical power at zero cost.

**The idea:** if we have a variable that correlates with the outcome 
*before* the experiment, we can strip out the noise it explains.

**The formula:**

$$Y_{cuped} = Y - \theta \cdot (X_{pre} - \mathbb{E}[X_{pre}])$$

$$\theta = \frac{Cov(Y, X_{pre})}{Var(X_{pre})}$$

- $Y$ = retention_7 (our outcome)
- $X_{pre}$ = sum_gamerounds (proxy for player engagement pre-experiment)
- $Y_{cuped}$ has the same mean as $Y$ but lower variance

**Variance reduction ≈ ρ²** where ρ is the correlation between 
sum_gamerounds and retention_7.

**Steps:**
1. Estimate θ on pooled data
2. Build Y_cuped for every player
3. Run the Z-test on Y_cuped

In [8]:
# Pooled data
pooled = pd.concat([control_balanced, treatment_balanced])

# Step 1 — Estimate theta
y     = pooled["retention_7"].astype(float)
x_pre = pooled["sum_gamerounds"].astype(float)

theta = np.cov(y, x_pre)[0, 1] / np.var(x_pre)
rho   = np.corrcoef(y, x_pre)[0, 1]

print(f"Theta (θ)             : {theta:.6f}")
print(f"Correlation (ρ)       : {rho:.4f}")
print(f"Variance reduction ≈ ρ²: {rho**2:.4f} ({rho**2*100:.1f}%)")

# Step 2 — Build Y_cuped
x_pre_mean = x_pre.mean()
pooled = pooled.copy()
pooled["retention_7_cuped"] = (
    pooled["retention_7"].astype(float) - theta * (pooled["sum_gamerounds"] - x_pre_mean)
)

# Step 3 — Z-test on Y_cuped
control_cuped   = pooled[pooled["version"] == "gate_30"]["retention_7_cuped"]
treatment_cuped = pooled[pooled["version"] == "gate_40"]["retention_7_cuped"]

from scipy.stats import ttest_ind

t_stat_cuped, p_cuped = ttest_ind(control_cuped, treatment_cuped, equal_var=False)

se_cuped = np.sqrt(control_cuped.var()/len(control_cuped) + treatment_cuped.var()/len(treatment_cuped))
diff_cuped = control_cuped.mean() - treatment_cuped.mean()
ci_low_cuped  = diff_cuped - 1.96 * se_cuped
ci_high_cuped = diff_cuped + 1.96 * se_cuped

print(f"\n--- CUPED Results ---")
print(f"Difference (CUPED)    : {diff_cuped:+.4f}")
print(f"95% CI (CUPED)        : [{ci_low_cuped:.4f}, {ci_high_cuped:.4f}]")
print(f"p-value (CUPED)       : {p_cuped:.4f}")

print(f"\n--- Comparison ---")
print(f"Original p-value      : 0.0029")
print(f"CUPED p-value         : {p_cuped:.4f}")
print(f"Original CI width     : {0.0129 - 0.0027:.4f}")
print(f"CUPED CI width        : {ci_high_cuped - ci_low_cuped:.4f}")

Theta (θ)             : 0.000554
Correlation (ρ)       : 0.2785
Variance reduction ≈ ρ²: 0.0775 (7.8%)

--- CUPED Results ---
Difference (CUPED)    : +0.0071
95% CI (CUPED)        : [0.0022, 0.0120]
p-value (CUPED)       : 0.0044

--- Comparison ---
Original p-value      : 0.0029
CUPED p-value         : 0.0044
Original CI width     : 0.0102
CUPED CI width        : 0.0098


### CUPED Interpretation

- **ρ = 0.28** → variance reduction of only 7.8%
- The CI narrowed slightly: from 0.0102 to 0.0098
- The p-value did not improve — and this is expected

**Important limitation:** CUPED requires a *pre-experiment* variable. 
`sum_gamerounds` was measured *during* the experiment (first 14 days), 
not before. This partially violates the CUPED assumption — the treatment 
itself may have influenced how many rounds players played.

In a production setting, we would use game rounds from the week *before* 
the experiment as X_pre. Without true pre-experiment data, CUPED gains 
are limited.

**The original Z-test conclusion stands:** gate_30 significantly retains 
more players (p = 0.0029, CI excludes zero).

## 9. Common Mistakes to Avoid

These are the most common errors in A/B testing and how we addressed them in this analysis.

| Mistake | What it is | How we handled it |
|---------|-----------|-------------------|
| **HARKing** | Changing the primary metric after seeing results | Fixed `retention_7` as primary metric before any analysis |
| **Peeking / p-hacking** | Checking p-value daily and stopping at significance | Sample size calculated upfront — no interim looks |
| **SRM ignored** | Not checking group balance before analyzing | Chi-squared test on group sizes — imbalance fixed by downsampling |
| **Wrong test** | Using Welch t-test for a binary metric | Z-test for proportions chosen based on metric type |
| **p < 0.05 = ship** | Treating significance as the only decision criterion | Checked effect size, CI width and guardrail metrics |
| **Guardrails ignored** | Shipping without checking secondary metrics | `sum_gamerounds` verified — no significant difference |
| **CUPED misapplied** | Using a variable measured during the experiment as X_pre | Flagged as limitation — true pre-experiment data needed |
| **Novelty effect** | Users behave differently at first out of curiosity | Dataset covers 14 days — enough for behavior to stabilize |
| **SUTVA violation** | One player's outcome depends on another's treatment | Players are independent — no network effects in a puzzle game |
| **Multiple comparisons** | Testing many metrics and picking the significant one | One predefined primary metric — secondary metrics reported separately |